## Generate outputs

Given that you have already run the Ingest and Run notebooks, this notebook takes the outputs of Run (the allpixels and allfires dataframes) and genarates the archival output files.

What we want out of this notebook is:
 - a snapshot of all the fires at a given t
 - a timeseries of each fire across time

In [1]:
import os
import datetime
import pandas as pd
import geopandas as gpd

from fireatlas import FireTime, FireObj, postprocess
from fireatlas.utils import timed

region = ['Quebec_PostHoc',]  # note you don't need the shape in here, just the name
tst = [2023, 4, 30, 'AM']
ted = [2023, 9, 15, 'PM']

2024-09-11 17:09:34,952 - fireatlas.FireLog - INFO - logger initialized!


## Read from disk

Since we want to use precisely the files that we just created in the Run notebook. We will set the `location` to "local".

In [2]:
allpixels = postprocess.read_allpixels(tst, ted, region, location="local")

2024-09-11 17:09:39,843 - fireatlas.FireLog - INFO - func:read_allpixels took: 1.59 sec


In [3]:
allfires_gdf = postprocess.read_allfires_gdf(tst, ted, region, location="local")

2024-09-11 17:09:40,690 - fireatlas.FireLog - INFO - func:read_allfires_gdf took: 840.88 ms


## Write snapshots

Write each geometry object into its own flatgeobuf file within a subdirectory.

In [4]:
%%time
postprocess.save_snapshots(allfires_gdf, region, tst, ted)

/srv/conda/envs/notebook/lib/python3.11/site-packages/geopandas/io/file.py:612: UserWarning: You are attempting to write an empty DataFrame to file. For some drivers, this operation may fail.
  _to_file_fiona(df, filename, driver, schema, crs, mode, **kwargs)
2024-09-11 17:14:06,232 - fireatlas.FireLog - INFO - func:save_snapshots took: 4.38 min


CPU times: user 3min 9s, sys: 3.07 s, total: 3min 12s
Wall time: 4min 23s


[]

## Write large fires

Start by getting a list containing all the fireIDs for the large fires in the allfires geodataframe.

In [5]:
large_fires = postprocess.find_largefires(allfires_gdf)

2024-09-11 17:14:06,250 - fireatlas.FireLog - INFO - func:find_largefires took: 7.35 ms


First we'll use the `allpixels` object to create the `nplist` layer

In [6]:
postprocess.save_large_fires_nplist(allpixels, region, large_fires, tst)

2024-09-11 17:14:57,272 - fireatlas.FireLog - INFO - func:save_large_fires_nplist took: 51.02 sec


The rest of the layers will be created directly from the `allfires_gdf`.

In [ ]:
postprocess.save_large_fires_layers(allfires_gdf, region, large_fires, tst, ted)